In [4]:
from google.colab import files
uploaded = files.upload()  # select your zip file

Saving hw-2-cn-ns-against-malaria (1).zip to hw-2-cn-ns-against-malaria (1).zip


In [5]:
!unzip "hw-2-cn-ns-against-malaria (1).zip" -d dataset/

Streaming output truncated to the last 5000 lines.
  inflating: dataset/dataset/train_images/C62P23N_ThinF_IMG_20150818_133211_cell_70.png  
  inflating: dataset/dataset/train_images/C62P23N_ThinF_IMG_20150818_133211_cell_76.png  
  inflating: dataset/dataset/train_images/C62P23N_ThinF_IMG_20150818_133211_cell_77.png  
  inflating: dataset/dataset/train_images/C62P23N_ThinF_IMG_20150818_133211_cell_86.png  
  inflating: dataset/dataset/train_images/C62P23N_ThinF_IMG_20150818_133211_cell_87.png  
  inflating: dataset/dataset/train_images/C62P23N_ThinF_IMG_20150818_133211_cell_94.png  
  inflating: dataset/dataset/train_images/C62P23N_ThinF_IMG_20150818_133307_cell_114.png  
  inflating: dataset/dataset/train_images/C62P23N_ThinF_IMG_20150818_133307_cell_129.png  
  inflating: dataset/dataset/train_images/C62P23N_ThinF_IMG_20150818_133307_cell_137.png  
  inflating: dataset/dataset/train_images/C62P23N_ThinF_IMG_20150818_133307_cell_141.png  
  inflating: dataset/dataset/train_images/C62

In [10]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset, random_split
from PIL import Image
import pandas as pd
import os
import time
import copy

train_csv  = '/content/dataset/dataset/train_data.csv'
train_path = '/content/dataset/dataset/train_images'
test_path  = '/content/dataset/dataset/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# CONFIG

IMG_SIZE = 128
BATCH_SIZE = 32
NUM_EPOCHS = 5
LEARNING_RATE = 0.001

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda
Epoch 1/5
train Loss: 0.3098 Acc: 0.8593
valid Loss: 0.1484 Acc: 0.9515

Epoch 2/5
train Loss: 0.1509 Acc: 0.9500
valid Loss: 0.1304 Acc: 0.9576

Epoch 3/5
train Loss: 0.1359 Acc: 0.9546
valid Loss: 0.1246 Acc: 0.9585

Epoch 4/5
train Loss: 0.1291 Acc: 0.9565
valid Loss: 0.1280 Acc: 0.9556

Epoch 5/5
train Loss: 0.1233 Acc: 0.9579
valid Loss: 0.1258 Acc: 0.9560

Training complete in 6m 31s
Submission shape: (5512, 2)
submission.csv saved successfully!


In [ ]:
# DATASET CLASS
class MalariaDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        if csv_path:
            self.df = pd.read_csv(csv_path)
        else:
            self.df = pd.DataFrame({
                "img_name": os.listdir(img_dir),
                "label": -1
            })
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.df.iloc[idx]["img_name"])
        image = Image.open(img_path).convert("RGB")
        label = self.df.iloc[idx]["label"]

        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
# Transformations

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [ ]:
# Load Data
train_dataset = MalariaDataset(train_csv, train_path, transform=train_transforms)
test_dataset = MalariaDataset(None, test_path, transform=test_transforms)

train_size = int(0.8 * len(train_dataset))
valid_size = len(train_dataset) - train_size

train_subset, valid_subset = random_split(train_dataset, [train_size, valid_size])

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_subset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

dataloaders = {
    "train": train_loader,
    "valid": valid_loader
}

In [ ]:
# Model Architecture
class CustomCNN(nn.Module):
    def __init__(self):
        super(CustomCNN, self).__init__()

        self.layer1 = self.conv_block(3, 64)
        self.layer2 = self.conv_block(64, 128)
        self.layer3 = self.conv_block(128, 256)
        self.layer4 = self.conv_block(256, 512)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 2)
        )

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.classifier(x)
        return x


model = CustomCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9)

In [ ]:
# Training
def train_model(model, dataloaders, criterion, optimizer, num_epochs):
    since = time.time()

    history = {
        "train_loss": [],
        "train_acc": [],
        "valid_loss": [],
        "valid_acc": []
    }

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")

        for phase in ["train", "valid"]:
            if phase == "train":
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device).long()

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)

            print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc.cpu().numpy())

        print()

    time_elapsed = time.time() - since
    print(f"Training complete in {time_elapsed//60:.0f}m {time_elapsed%60:.0f}s")

    return model, history

    model, history = train_model(model, dataloaders, criterion, optimizer, NUM_EPOCHS)

In [ ]:
# Test Predictions
model.eval()
predictions = []
filenames = []

with torch.no_grad():
    for i, (inputs, _) in enumerate(test_loader):
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        predictions.append(preds.item())
        filenames.append(test_dataset.df.iloc[i]["img_name"])

In [ ]:
submission = pd.DataFrame({"img_name": filenames,"label": predictions})
print("Submission shape:", submission.shape)  # should be (5512, 2)
submission.to_csv("submission.csv", index=False)
print("submission.csv saved successfully!")